# 실습 1: 협업 필터링 파이프라인 Overview (User-based CF 한 번에 실행)

이 실습은 MovieLens 데이터를 불러와 User-based 협업 필터링 추천 시스템을 **처음부터 끝까지** 한 번에 체험하는 오버뷰(Overview) 실습입니다.

**개념 복기 및 이론 점검**
- '나와 비슷한 사용자가 좋아한 것을 나도 좋아할 것이다'는 직관이 코드에서 어떻게 구현되는지 관찰합니다.
- 평점 데이터로 만든 User-Item 행렬이 어떤 구조인지, 왜 대부분이 0인지 살펴봅니다.
- 코사인 유사도가 '비슷한 사용자 찾기'에 어떤 역할을 하는지 확인합니다.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Team-AnI/A-AND-I-4TH-AI-CODE-LAB/blob/main/4주차/lab_01_pipeline.ipynb)

> 위 배지를 누르면 이 노트북이 **여러분 Google 계정의 Colab**에서 열립니다. 수정본을 남기려면 `파일 → Drive에 사본 저장`.

## 1. 데이터 다운로드
MovieLens ml-100k 데이터셋을 내려받습니다.

In [ ]:
!wget -q https://files.grouplens.org/datasets/movielens/ml-100k.zip -O ml-100k.zip
!unzip -q -o ml-100k.zip
print("다운로드 완료")

## 2. 라이브러리 임포트 및 환경 설정

In [ ]:
import pandas as pd
import torch
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)

## 3. 데이터 로드 및 User-Item 평점 행렬 구성
943명의 사용자와 1682편의 영화로 이루어진 평점 행렬을 만듭니다. 대부분의 칸이 0(미평가)인 희소(sparse) 구조입니다.

In [ ]:
df = pd.read_csv(
    'ml-100k/u.data', sep='\t', header=None,
    names=['user_id', 'item_id', 'rating', 'timestamp']
)

movies = pd.read_csv(
    'ml-100k/u.item', sep='|', header=None, encoding='latin-1',
    usecols=[0, 1], names=['item_id', 'title']
)
movie_names = dict(zip(movies['item_id'], movies['title']))

n_users = df['user_id'].max()
n_items = df['item_id'].max()

ratings = torch.zeros(n_users, n_items)
for row in df.itertuples():
    ratings[row.user_id - 1, row.item_id - 1] = row.rating

print(f"사용자 수: {n_users}, 영화 수: {n_items}")
print(f"총 평점 수: {len(df)}")
print(f"행렬 크기: {ratings.shape}")

## 4. 행렬 구조 시각화
처음 30명 사용자 × 처음 50편 영화의 평점 분포를 시각화합니다. 흰 칸이 많을수록 행렬이 희소합니다.

In [ ]:
subset = ratings[:30, :50].numpy()

plt.figure(figsize=(12, 5))
plt.imshow(subset, aspect='auto', cmap='YlOrRd', interpolation='nearest')
plt.colorbar(label='Rating (0 = unrated)')
plt.title('User-Item Rating Matrix (first 30 users × 50 movies)')
plt.xlabel('Movie ID')
plt.ylabel('User ID')
plt.tight_layout()
plt.show()

sparsity = 1 - (df.shape[0] / (n_users * n_items))
print(f"행렬 희소도(Sparsity): {sparsity:.2%}")

## 5. 🔑 핵심 실습: 코사인 유사도로 사용자 간 유사도 계산
각 사용자의 평점 벡터를 단위 벡터로 정규화한 뒤, 행렬 곱으로 모든 사용자 쌍의 코사인 유사도를 한 번에 계산합니다.

In [ ]:
norms = torch.norm(ratings, dim=1, keepdim=True).clamp(min=1e-8)
normalized = ratings / norms
user_sim = torch.mm(normalized, normalized.T)  # (943, 943)

print(f"유사도 행렬 크기: {user_sim.shape}")
print(f"사용자 0과 사용자 1의 유사도: {user_sim[0, 1].item():.4f}")
print(f"사용자 0과 자기 자신의 유사도: {user_sim[0, 0].item():.4f}")

## 6. User-based CF — 이웃 선정 및 추천 생성

In [ ]:
def recommend_user_based(user_idx, ratings, user_sim, top_n_neighbors=20, top_n_items=10):
    sim_scores = user_sim[user_idx].clone()
    sim_scores[user_idx] = -1  # 자기 자신 제외

    top_neighbors = torch.topk(sim_scores, top_n_neighbors).indices

    unrated_mask = (ratings[user_idx] == 0)
    neighbor_ratings = ratings[top_neighbors]           # (top_n_neighbors, n_items)
    weights = sim_scores[top_neighbors].unsqueeze(1)    # (top_n_neighbors, 1)

    scores = (neighbor_ratings * weights).sum(dim=0)    # (n_items,)
    scores[~unrated_mask] = -float('inf')               # 이미 평가한 영화 제외

    return torch.topk(scores, top_n_items).indices.tolist()


user_idx = 0
rec_items = recommend_user_based(user_idx, ratings, user_sim)

print(f"사용자 {user_idx + 1}번의 추천 영화 Top 10:")
for rank, item_idx in enumerate(rec_items, 1):
    print(f"  {rank}. {movie_names.get(item_idx + 1, 'Unknown')}")

## 7. 사용자별 유사도 분포 시각화
사용자 0번에 대한 다른 모든 사용자의 유사도 값이 어떻게 분포하는지 확인합니다.

In [ ]:
sim_values = user_sim[0].clone()
sim_values[0] = float('nan')  # 자기 자신 제외

plt.figure(figsize=(8, 4))
plt.hist(sim_values.numpy()[~np.isnan(sim_values.numpy())], bins=50, color='steelblue', edgecolor='white')
plt.title('Cosine Similarity Distribution (User 1 vs. All Others)')
plt.xlabel('Cosine Similarity')
plt.ylabel('Count')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. ✅ 학습 결과 정리
- MovieLens ml-100k 데이터를 User-Item 평점 행렬로 변환하고, 그 구조(희소도)를 시각적으로 확인했습니다.
- 각 사용자의 평점 벡터를 코사인 유사도로 비교해, '나와 비슷한 사용자'를 수치로 찾아냈습니다.
- 유사한 이웃들의 평점을 가중합산해 추천 목록을 생성하는 User-based CF의 전체 흐름을 실행했습니다.
- 🎯 **핵심 결론:** 코드가 어떻게 돌아가는지 눈에 익혔다면 성공입니다! 다음 실습(lab_02)에서는 이 코드를 **한 줄씩 분해하여 왜 그렇게 짜여 있는지** 깊게 알아봅니다.